# GPU Notebook — Arithmetic (Addition) Interpretability

Companion to `run_gpu.ipynb`, targeting the **addition** behaviour (carry logic).

**Key difference:** the SAEs were trained on *pooled* activations from all three
behaviours (capitals + addition + units), so they already cover arithmetic. This
notebook **reuses the existing SAE checkpoints** and therefore **skips activation
capture and SAE training entirely** — the ~50 min/prompt training cost is not
repeated here.

### Per-digit attribution (important)
Qwen tokenizes numbers **one digit at a time**, most-significant first, and the
prompt must end with a **trailing space** after `Answer:` (otherwise the model's
top prediction is just a space, and attribution has no signal). We therefore build
a **separate attribution graph for each answer digit**, teacher-forcing the
already-emitted digits into the prompt. This is what makes carry logic visible —
a single whole-answer graph only attributes to the first digit.

### What to expect
Baseline analysis shows the model is **~98% confident on every addition prompt**
(even the hardest carry cases). That means *ablation* effects will likely be small
(there is little uncertainty to remove). The **swap** experiments (carry ↔ no-carry)
at the tens digit are the most likely to produce a visible, interpretable shift.

**Runtime settings:** GPU (any tier), High-RAM recommended.


## Step 1 — Mount Drive & extract project

In [ ]:
# Step 1: Mount Google Drive and extract project
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Set to True if you uploaded project.zip directly to the Colab files sidebar:
uploaded_directly_to_colab = False

if uploaded_directly_to_colab:
    zip_path = "/content/project.zip"
else:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    !unzip -q -o {zip_path} -d /content/
    os.chdir('/content')
    print(f"Current working directory: {os.getcwd()}")
    !ls -l
else:
    print(f"ERROR: Could not find {zip_path}.")
    print("Please upload 'project.zip' to Google Drive or the Colab files sidebar.")

## Step 2 — Install dependencies

In [ ]:
# Step 2: Install Dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .

## Step 3 — Generate the addition dataset
Only the arithmetic CSV is needed here.

In [ ]:
# Step 3: Generate Datasets (addition CSV is what we need)
!python data/generate_datasets.py
print("\nDataset files:")
!ls -lh data/addition_data.csv

## Step 4 — Restore SAE checkpoints from Drive (no training)

The SAEs are behaviour-agnostic (trained on pooled activations), so we reuse the
checkpoints produced by `run_gpu.ipynb`. This cell copies them back from Drive into
the local workspace. **If they are missing**, run `run_gpu.ipynb` Step 5 first (or
uncomment the training fallback below).

In [ ]:
# Step 4: Restore SAE checkpoints from Drive
import os, shutil, glob

drive_sae = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints'
local_sae = '/content/mechanistic_data/sae_checkpoints'
os.makedirs(local_sae, exist_ok=True)

if os.path.isdir(drive_sae) and glob.glob(f'{drive_sae}/sae_layer*.pt'):
    for f in os.listdir(drive_sae):
        shutil.copy2(f'{drive_sae}/{f}', local_sae)
    print("Restored SAE checkpoints from Drive:")
    !ls -lh {local_sae}
else:
    print("No SAE checkpoints found in Drive at:", drive_sae)
    print("Run run_gpu.ipynb Step 5 (training) first, or uncomment the fallback below.")
    # --- Training fallback (slow): captures activations then trains all 8 layers ---
    # !python src/train.py --config configs/sae_config.yaml \
    #   --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints

---
## Attribution graphs — carry-arithmetic prompts (per-digit)

Qwen tokenizes numbers **one digit at a time** and emits them **most-significant
first**. So the answer `141` is three decode steps: `1` → `4` → `1`. We build a
**separate attribution graph for each digit**, teacher-forcing the already-emitted
digits into the prompt:

| Digit | Prompt (teacher-forced) | Target | What it reveals |
|-------|-------------------------|--------|-----------------|
| hundreds | `... Answer: `   | `1` | carry *propagating out* (magnitude) |
| tens     | `... Answer: 1`  | `4` | carry being *consumed* (5+8+**1**=14) |
| units    | `... Answer: 14` | `1` | carry being *generated* (8+3=11) |

**The units and tens graphs are the carry-relevant ones** — the units digit is where
the carry is born, the tens digit is where it gets added in. The hundreds digit is
just the carry propagating out, but it's still a valid "did the model resolve the
magnitude" check.

Each graph still prints a ready-to-paste `--features` string, and the downstream
intervention cells point at the **tens-digit** graph (the carry-consuming step).


In [ ]:
# Graph 1: 58 + 83 = 141  (carry) -- one graph per digit
# hundreds digit: 1  (carry propagating out)
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: " \
  --target "1" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_58_83_h_graph.json \
  --output-html outputs/add_58_83_h_graph.html \
  --output-mermaid outputs/add_58_83_h_graph.md

# tens digit: 4  (carry consumed: 5+8+1=14)  <-- key carry graph
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target "4" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_58_83_t_graph.json \
  --output-html outputs/add_58_83_t_graph.html \
  --output-mermaid outputs/add_58_83_t_graph.md

# units digit: 1  (carry generated: 8+3=11)
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: 14" \
  --target "1" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_58_83_u_graph.json \
  --output-html outputs/add_58_83_u_graph.html \
  --output-mermaid outputs/add_58_83_u_graph.md

In [ ]:
# Graph 2: 76 + 98 = 174  (carry) -- one graph per digit
# hundreds digit: 1  (carry propagating out)
!python src/attribution_graph.py \
  --prompt "Question: What is 76 + 98? Answer: " \
  --target "1" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_76_98_h_graph.json \
  --output-html outputs/add_76_98_h_graph.html \
  --output-mermaid outputs/add_76_98_h_graph.md

# tens digit: 7  (carry consumed: 7+9+1=17)  <-- key carry graph
!python src/attribution_graph.py \
  --prompt "Question: What is 76 + 98? Answer: 1" \
  --target "7" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_76_98_t_graph.json \
  --output-html outputs/add_76_98_t_graph.html \
  --output-mermaid outputs/add_76_98_t_graph.md

# units digit: 4  (carry generated: 6+8=14)
!python src/attribution_graph.py \
  --prompt "Question: What is 76 + 98? Answer: 17" \
  --target "4" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_76_98_u_graph.json \
  --output-html outputs/add_76_98_u_graph.html \
  --output-mermaid outputs/add_76_98_u_graph.md

In [ ]:
# Graph 3: 98 + 79 = 177  (carry) -- one graph per digit
# hundreds digit: 1  (carry propagating out)
!python src/attribution_graph.py \
  --prompt "Question: What is 98 + 79? Answer: " \
  --target "1" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_98_79_h_graph.json \
  --output-html outputs/add_98_79_h_graph.html \
  --output-mermaid outputs/add_98_79_h_graph.md

# tens digit: 7  (carry consumed: 9+7+1=17)  <-- key carry graph
!python src/attribution_graph.py \
  --prompt "Question: What is 98 + 79? Answer: 1" \
  --target "7" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_98_79_t_graph.json \
  --output-html outputs/add_98_79_t_graph.html \
  --output-mermaid outputs/add_98_79_t_graph.md

# units digit: 7  (carry generated: 8+9=17)
!python src/attribution_graph.py \
  --prompt "Question: What is 98 + 79? Answer: 17" \
  --target "7" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_98_79_u_graph.json \
  --output-html outputs/add_98_79_u_graph.html \
  --output-mermaid outputs/add_98_79_u_graph.md

### Checkpoint: copy graphs to Drive

In [ ]:
# Persist attribution graphs immediately (crash-safe)
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for f in glob.glob('/content/outputs/add_*graph*'):
    shutil.copy2(f, drive_out)
    print("Copied", os.path.basename(f))

---
## Diagnostic 1 — Full MLP knockout

Upper bound on how much the hooked layers matter. Given ~98% baseline confidence,
expect this to move the prediction only a little (unlike the low-confidence capitals
prompts). If it does *nothing*, the addition computation isn't in these MLP outputs
at the last token.

In [ ]:
# Full knockout at the TENS digit: 58 + 83, Answer: 1 -> 4 (carry-consuming step)
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4" \
  --full-knockout \
  --output outputs/add_knockout_58_83.json

## Diagnostic 2 — Progressive ablation scan
Ablate the top-10/25/50/100/all graph features and watch the logit of the answer.

In [ ]:
# Scan the TENS-digit graph: 58 + 83, Answer: 1 -> 4
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4" \
  --graph-json outputs/add_58_83_t_graph.json \
  --scan \
  --output outputs/add_scan_58_83.json

## Experiment 1 — Targeted inhibition (all graph features)

In [ ]:
# Inhibit all tens-digit graph features: 76 + 98, Answer: 1 -> 7
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 76 + 98? Answer: 1" \
  --target-token "7" \
  --graph-json outputs/add_76_98_t_graph.json \
  --output outputs/add_inhibit_76_98.json

## Experiment 2 — Activation swap-in at the tens digit (the promising one)

Swap features from a **no-carry** source into the **carry** target, both teacher-forced
so the *next* token is the **tens digit** — the position where an incoming carry is
either added or dropped:

- **Target (carry):** `58 + 83`, prompt `... Answer: 1`. Correct tens digit is `4`
  (5+8+**1**=14). If the carry is dropped, the tens digit becomes `3` (5+8=13).
- **Source (no-carry):** `32 + 42`, prompt `... Answer: ` — its tens column (3+4=7)
  has **no incoming carry**, so its activations carry a "no-carry" signature.

We track `4` (carry kept, correct) vs `3` (carry dropped). Because Qwen tokenizes
digits individually and `4` / `3` are now **distinct single tokens**, this contrast
is finally measurable — the previous `141` vs `131` targets both collapsed to the
first token `1` and were indistinguishable.

**Caveat:** the operands differ between source and target, so the swap is a coarse
probe, not a clean isolate — a shift toward `3` is *suggestive* of a carry feature,
to be confirmed by the inhibition/scan results on the tens-digit graph.


In [ ]:
# Swap at TENS digit: no-carry (32+42=74, tens=7) source -> carry (58+83, tens=4) target
# Both teacher-forced to the tens position. Track 4 (carry kept) vs 3 (carry dropped).
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 32 + 42? Answer: " \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --graph-json outputs/add_58_83_t_graph.json \
  --target-token "4, 3" \
  --output outputs/add_swap_nocarry_to_carry.json

In [ ]:
# Full latent swap at TENS digit (no --features / --graph-json) -- swaps everything
# Track 4 (carry kept), 3 (carry dropped), 7 (source's own tens digit, 3+4=7)
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 32 + 42? Answer: " \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --output outputs/add_swap_full_nocarry_to_carry.json

---
## Copy all outputs to Drive

In [ ]:
# Copy all addition outputs to Google Drive for persistence
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/outputs/add_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')

---
## Notes on interpreting these results

- **Per-digit is necessary, not optional.** Qwen emits numbers one digit at a time,
  most-significant first. A single graph on the whole answer only ever attributes to
  the *first* digit, and any target that differs only in a later digit (e.g. `141`
  vs `131`) collapses to the same token. Each digit gets its own graph + teacher-forced
  prompt so carrying is actually visible.
- **Where carry lives:** units digit = carry *generated* (e.g. 8+3=11), tens digit =
  carry *consumed* (5+8+**1**=14), hundreds digit = carry *propagated out*. The tens
  and units graphs are the carry-relevant ones; the interventions target the tens digit.
- **"Did the model get it right?"** is answered by `baseline.py` (full greedy
  generation), NOT by a single attribution graph. A high top-1 prob on each digit
  (e.g. `1`@0.93, then `4`, then `1`) is corroborating evidence, but report accuracy
  from the baseline.
- **Low ablation effect is an expected result, not a bug.** The model is ~98%
  confident, so there is little logit mass for ablation to move. Report the
  knockout/scan deltas as evidence of *where* the arithmetic is computed.
- **The swap is the informative experiment.** A shift toward the dropped-carry
  digit (`3` instead of `4`) under a no-carry swap is direct evidence of a
  carry-handling feature — but note operands differ between source and target, so
  confirm against the inhibition/scan on the tens-digit graph.
- **How this maps to Anthropic's addition circuit** (*On the Biology of a Large
  Language Model*, 2025): they found addition uses parallel "magnitude" (~coarse
  sum) and "ones-digit lookup" pathways that combine. Per-digit graphs are what let
  you look for those two feature families separately here.
- **Layer 32's SAE is degenerate** (val_mse ≈ 0.12 vs ≈ 0.002 at layer 4). If a
  result hinges on layer-32 features, treat it skeptically or drop layer 32 via
  `--layers 4 8 12 16 20 24 28`.
